- 北京大学金融工程实验室《人工智能与财务分析》课件

## 第2讲、AI财务分析


以下每个cell的内容可存为相关.py文件，然后整合在一起运行。 



In [1]:
# get financial statement

import pandas as pd
import os
import sys

DATA_DIR = r"/usr/local/data"
VALID_YEARS = [2022, 2023, 2024]
FINANCIAL_TYPES = {
    "资产负债表": "资产负债表",
    "利润表": "利润表",
    "现金流量表": "现金流量表"
}

_cache = {}

def get_file_path(year, financial_type):
    return os.path.join(DATA_DIR, f"{year}年{financial_type}.xlsx")

def extract_indicator_name(header):
    if pd.isna(header):
        return ""
    return str(header).split('\n')[0].strip()


def load_financial_data(year, financial_type):
    cache_key = f"{year}_{financial_type}"
    if cache_key in _cache:
        return _cache[cache_key]
    
    file_path = get_file_path(year, financial_type)
    if not os.path.exists(file_path):
        return None
    df = pd.read_excel(file_path)
    _cache[cache_key] = df
    return df


def search_security(df, security_name):
    if df is None:
        return []
    matches = df[df['证券名称'] == security_name]
    return matches.index.tolist()


def format_output(company_row):
    if company_row is None:
        return "未找到数据"

    result = []
    result.append(f"证券代码: {company_row['证券代码']}")
    result.append(f"证券名称: {company_row['证券名称']}")
    
    for col in company_row.index:
        if col not in ['证券代码', '证券名称']:
            value = company_row[col]
            if pd.notna(value):
                clean_name = extract_indicator_name(col)
                result.append(f"{clean_name}: {value}")

    return "\n".join(result)


def validate_year(year_str):
    try:
        year = int(year_str)
        if year in VALID_YEARS:
            return year
    except ValueError:
        pass
    return None


def validate_financial_type(ftype_str):
    for key in FINANCIAL_TYPES:
        if key in ftype_str:
            return FINANCIAL_TYPES[key]
    return None


def main():
    print("=" * 50)
    print("财务表查询系统")
    print("=" * 50)

    security_name = input("请输入证券名称：").strip()
    if not security_name:
        print("证券名称不能为空")
        return

    year_str = input("请输入年份（2022-2024）：").strip()
    year = validate_year(year_str)
    if year is None:
        print(f"年份无效，请输入 {VALID_YEARS} 中的一个")
        return

    financial_type_str = input("请输入财务表类型（资产负债表/利润表/现金流量表）：").strip()
    financial_type = validate_financial_type(financial_type_str)
    if financial_type is None:
        print("财务表类型无效，请输入：资产负债表、利润表 或 现金流量表")
        return

    df = load_financial_data(year, financial_type)
    if df is None:
        print(f"未找到 {year}年{financial_type} 的数据文件")
        return

    indices = search_security(df, security_name)

    if len(indices) == 0:
        print(f"未找到证券名称为 '{security_name}' 的记录")
        return
    elif len(indices) > 1:
        print(f"找到多个匹配的证券，请选择：")
        for i, idx in enumerate(indices):
            print(f"{i+1}. {df.iloc[idx]['证券名称']} ({df.iloc[idx]['证券代码']})")
        choice = input("请输入序号选择（直接回车退出）：").strip()
        if not choice:
            return
        try:
            choice_idx = int(choice) - 1
            if 0 <= choice_idx < len(indices):
                row_idx = indices[choice_idx]
            else:
                print("无效选择")
                return
        except ValueError:
            print("无效输入")
            return
    else:
        row_idx = indices[0]

    company_row = df.iloc[row_idx]

    print("\n" + "=" * 50)
    print(f"{year}年{financial_type}")
    print("=" * 50)
    print(format_output(company_row))


# if __name__ == "__main__":
#     main()


In [2]:
"""LLM 调用接口

基于 DeepSeek API 的大语言模型调用工具

依赖:
    openai: pip install openai

使用方法:
    from LLM import chat

    # 简单调用
    result = chat("你好，请介绍一下自己")

    # 带 system prompt
    result = chat("你是一个专业的金融分析师", "分析一下同花顺2022年的年报")

    # 带历史消息
    messages = [
        {"role": "system", "content": "你是一个助手"},
        {"role": "user", "content": "你好"}
    ]
    result = chat_with_history(messages, "今天天气怎么样")
"""

import os
from typing import Optional, List, Dict
from openai import OpenAI

DEFAULT_API_KEY = "sk-d7b49c4ab79e4a8095216feae3541b2d"
DEFAULT_BASE_URL = "https://api.deepseek.com"
DEFAULT_MODEL = "deepseek-chat"


class LLMClient:
    """LLM 客户端"""

    def __init__(
        self,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        model: Optional[str] = None
    ):
        self.api_key = api_key or os.environ.get("DEEPSEEK_API_KEY", DEFAULT_API_KEY)
        self.base_url = base_url or os.environ.get("DEEPSEEK_BASE_URL", DEFAULT_BASE_URL)
        self.model = model or DEFAULT_MODEL
        self.client = OpenAI(api_key=self.api_key, base_url=self.base_url)

    def chat(
        self,
        user_message: str,
        system_prompt: Optional[str] = None,
        temperature: float = 0.7,
        stream: bool = False
    ) -> Dict:
        """简单的单轮对话

        Args:
            user_message: 用户消息
            system_prompt: 系统提示词（可选）
            temperature: 温度参数 (0-2)
            stream: 是否流式输出

        Returns:
            包含回复内容和token使用信息的字典
        """
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": user_message})

        return self._create_completion(messages, temperature, stream)

    def chat_with_history(
        self,
        messages: List[Dict[str, str]],
        new_message: str,
        temperature: float = 0.7,
        stream: bool = False
    ) -> Dict:
        """带历史消息的多轮对话

        Args:
            messages: 历史消息列表
            new_message: 新增的用户消息
            temperature: 温度参数 (0-2)
            stream: 是否流式输出

        Returns:
            包含回复内容和token使用信息的字典
        """
        messages = messages + [{"role": "user", "content": new_message}]
        return self._create_completion(messages, temperature, stream)

    def _create_completion(
        self,
        messages: List[Dict[str, str]],
        temperature: float,
        stream: bool
    ) -> Dict:
        """创建对话完成"""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            stream=stream
        )

        if stream:
            return response
        
        usage = response.usage
        return {
            "content": response.choices[0].message.content,
            "prompt_tokens": usage.prompt_tokens if usage else 0,
            "completion_tokens": usage.completion_tokens if usage else 0,
            "total_tokens": usage.total_tokens if usage else 0
        }


_default_client = None


def get_client() -> LLMClient:
    """获取默认客户端实例"""
    global _default_client
    if _default_client is None:
        _default_client = LLMClient()
    return _default_client


def chat(
    system_prompt: Optional[str] = None,
    user_message: Optional[str] = None,
    temperature: float = 0.7,
    stream: bool = False
) -> Dict:
    """简单的对话接口

    Args:
        system_prompt: 系统提示词（可选）
        user_message: 用户消息
        temperature: 温度参数 (0-2)
        stream: 是否流式输出

    Returns:
        包含回复内容和token使用信息的字典: {"content": str, "prompt_tokens": int, "completion_tokens": int, "total_tokens": int}

    使用示例:
        # 方式1: 只有用户消息
        result = chat(user_message="你好")

        # 方式2: 有系统提示和用户消息
        result = chat("你是一个诗人", "写一首关于春天的诗")
    """
    # 兼容旧调用方式：如果第一个参数不是 system_prompt（是 user_message）
    if system_prompt and user_message is None:
        user_message = system_prompt
        system_prompt = None
    elif system_prompt and user_message:
        pass  # 正常情况
    elif user_message is None:
        raise ValueError("user_message cannot be None")

    client = get_client()
    return client.chat(user_message, system_prompt, temperature, stream)


def chat_with_history(
    messages: List[Dict[str, str]],
    new_message: str,
    temperature: float = 0.7,
    stream: bool = False
) -> str:
    """带历史消息的对话接口

    Args:
        messages: 历史消息列表
        new_message: 新增的用户消息
        temperature: 温度参数 (0-2)
        stream: 是否流式输出

    Returns:
        LLM 回复内容

    使用示例:
        messages = [
            {"role": "system", "content": "你是一个助手"},
            {"role": "user", "content": "你好"}
        ]
        result = chat_with_history(messages, "今天天气怎么样")
    """
    client = get_client()
    return client.chat_with_history(messages, new_message, temperature, stream)


# if __name__ == "__main__":
#     # 简单测试
#     print("=== 测试1: 只有用户消息 ===")
#     result = chat(user_message="你好")
#     print(result)

#     print("\n=== 测试2: 有系统提示 ===")
#     result = chat("你是一个诗人", "写一首关于春天的诗")
#     print(result)

#     print("\n=== 测试3: 带历史消息 ===")
#     messages = [
#         {"role": "system", "content": "你是一个专业的金融分析师"},
#         {"role": "user", "content": "同花顺的主营业务是什么？"}
#     ]
#     result = chat_with_history(messages, "2022年它的营收情况如何？")
#     print(result)


In [3]:
"""年报管理层讨论与分析查询工具

该工具用于查询上市公司年报中的"管理层讨论与分析"(MDA)部分。

依赖:
    - requests
    - pandas (可选，用于数据处理)

使用方法:
    # 方式1: 通过环境变量设置授权码
    import os
    os.environ["MCP_AUTHORIZATION"] = "your_auth_token"
    from get_mda import get_mda
    result = get_mda("同花顺", 2022)

    # 方式2: 直接传入授权码
    from get_mda import get_mda
    result = get_mda("同花顺", 2022, authorization="your_auth_token")

    # 打印格式化结果
    from get_mda import format_mda_output
    print(format_mda_output(result, "同花顺", 2022))
"""

import os
import sys
import json
import requests
from typing import Optional, Dict, Any

os.environ["MCP_AUTHORIZATION"] = "eyJhbGciOiJSU0EtT0FFUC0yNTYiLCJlbmMiOiJBMjU2R0NNIn0.Rioo8gCj5ENA66qH0AHQdXa1eURB0VLwmp0p4yNP4kPN8ZaW1XEHZkbQd2CO7afFqbm5iCEbFXHWSc4eIcbHRn7ilusLjCESC59wj4WSh92hVthGLR4Ytv_Ap3RhIQXTsCCQImm7lqWkJ1JkdP5NX32psX8r-wlJ26wqTfHXWO1EAeFJHM_piAkXadzsmvlgMAgsUZiZRwXIUCwC_3RyMN4Atk4aUo9ec6Yyrx-KLOvKYkqbk0oVZgugeNu8eX67wsqiYFyEgk5aP5AwQnjXFHy2K_HVE577iD559_5fvntAgD7RButF1lYn8ZS-Wfzk5-BgktFo_0PXa7Ld2bIpNg.F2KiKBPMExPB1iAr.JYPx-7lxmT4MFhWC2OCp_UJL0_bF6cEtEnkBCqD7UGeJxrAuCNlDUJ2Q-nPplnCCRaXcdjDSQaeTGK-AG-SqmIBG9lfnKuVWuXvuFVJYHv53qHyzbsywVi5CBViM1Hu--7vNUcN2-9IdySLlZHj6H7J260SbgUgl72A9lro4ZIZUy5nA2672vXXMsre3nAbGhGHJPqvGU_VGLcLRu6gssnuymW6-rkfcdrTn_QRLm3UfmOhU-qGqmhc_.vgEfJUtdaZ7kNa3HjQAN6A"
class MCPClient:
    """MCP HTTP客户端"""

    STOCK_MCP_URL = "https://api-mcp.51ifind.com:8643/ds-mcp-servers/hexin-ifind-ds-stock-mcp"
    FUND_MCP_URL = "https://api-mcp.51ifind.com:8643/ds-mcp-servers/hexin-ifind-ds-fund-mcp"
    EDB_MCP_URL = "https://api-mcp.51ifind.com:8643/ds-mcp-servers/hexin-ifind-ds-edb-mcp"
    NEWS_MCP_URL = "https://api-mcp.51ifind.com:8643/ds-mcp-servers/hexin-ifind-ds-news-mcp"

    def __init__(
        self,
        url: str,
        authorization: str,
        timeout: int = 30,
        headers: Optional[Dict[str, str]] = None
    ):
        self.url = url
        self.authorization = authorization
        self.timeout = timeout
        self.default_headers = {
            "Authorization": authorization,
            "Content-Type": "application/json; charset=utf-8"
        }
        if headers:
            self.default_headers.update(headers)

    def call_tool(self, tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
        """调用MCP工具"""
        payload = {
            "jsonrpc": "2.0",
            "id": 1,
            "method": "tools/call",
            "params": {
                "name": tool_name,
                "arguments": arguments
            }
        }

        response = requests.post(
            self.url,
            headers=self.default_headers,
            json=payload,
            timeout=self.timeout
        )

        result = response.json()

        if "result" in result and "content" in result["result"]:
            text = result["result"]["content"][0].get("text", "{}")
            try:
                return json.loads(text)
            except json.JSONDecodeError:
                return {"data": text}
        elif "error" in result:
            return {"error": result["error"]}

        return result


class NewsMCP:
    """新闻资讯MCP工具类"""

    def __init__(self, client: Optional[MCPClient] = None, authorization: Optional[str] = None):
        if client is None:
            auth = authorization or os.getenv("MCP_AUTHORIZATION")
            if not auth:
                raise ValueError("MCP_AUTHORIZATION is required. Set it as environment variable or pass it as parameter.")
            client = MCPClient(url=MCPClient.NEWS_MCP_URL, authorization=auth)
        self.client = client

    def search_notice(
        self,
        query: str,
        size: int = 5,
        time_start: str = "2025-01-01",
        time_end: str = "2026-01-01"
    ) -> Dict[str, Any]:
        """公告内容语义查询"""
        return self.client.call_tool(
            "search_notice",
            {
                "query": query,
                "size": size,
                "time_start": time_start,
                "time_end": time_end
            }
        )


def get_mda(company_name: str, year: int, authorization: Optional[str] = None) -> Dict[str, Any]:
    """查询上市公司年报管理层讨论与分析

    Args:
        company_name: 上市公司名称
        year: 年份
        authorization: MCP授权码 (必须设置，可通过环境变量 MCP_AUTHORIZATION 或参数传入)

    Returns:
        包含年报管理层讨论与分析的字典

    Raises:
        ValueError: 当未设置授权码时
    """
    auth = authorization or os.getenv("MCP_AUTHORIZATION")
    if not auth:
        raise ValueError(
            "MCP_AUTHORIZATION is required. "
            "Please set it as environment variable or pass it as parameter. "
            "Example: os.environ['MCP_AUTHORIZATION'] = 'your_token'"
        )

    news_mcp = NewsMCP(authorization=auth)

    query = f"{company_name} {year}年度报告"

    try:
        result = news_mcp.search_notice(
            query=query,
            size=20,
            time_start=f"{year - 1}-11-01",
            time_end=f"{year + 1}-05-31"
        )

        if "data" in result and "data" in result["data"]:
            data_str = result["data"]["data"]
            if isinstance(data_str, str):
                result["data"]["data"] = json.loads(data_str)

        return result
    except Exception as e:
        return {"error": str(e)}


def format_mda_output(result: Dict[str, Any], company_name: str, year: int) -> str:
    """格式化输出管理层讨论与分析内容

    Args:
        result: 查询结果
        company_name: 公司名称
        year: 年份

    Returns:
        格式化后的字符串
    """
    if "error" in result:
        return f"查询出错: {result['error']}"

    try:
        data = result.get("data", {}).get("data", [])
    except Exception:
        data = []

    if not data:
        return f"未找到 {company_name} {year}年年度报告的管理层讨论与分析"

    output_lines = []
    output_lines.append("=" * 60)
    output_lines.append(f"{company_name} {year}年年度报告 - 管理层讨论与分析")
    output_lines.append("=" * 60)

    found = False
    for item in data:
        title = item.get("公告标题", "")
        if str(year) in title and "年度报告" in title:
            found = True
            output_lines.append(f"\n公告标题: {title}")
            output_lines.append(f"公告日期: {item.get('日期', '')}")
            output_lines.append("-" * 60)
            content = item.get("公告片段内容", "")
            if content:
                output_lines.append("\n管理层讨论与分析:")
                output_lines.append(content)
            else:
                output_lines.append("\n未找到相关内容")

    if not found:
        output_lines.append(f"\n未找到 {company_name} {year}年年度报告")

    return "\n".join(output_lines)


def validate_year(year_str: str) -> Optional[int]:
    """验证年份输入

    Args:
        year_str: 年份字符串

    Returns:
        有效的年份整数，或None
    """
    try:
        year = int(year_str)
        if 2000 <= year <= 2030:
            return year
    except ValueError:
        pass
    return None


def main():
    print("=" * 60)
    print("年报管理层讨论与分析查询系统")
    print("=" * 60)

    auth = os.environ.get("MCP_AUTHORIZATION")
    if not auth:
        print("错误: 请先设置环境变量 MCP_AUTHORIZATION")
        print("或者在代码中设置: os.environ['MCP_AUTHORIZATION'] = 'your_token'")
        return

    company_name = input("请输入上市公司名称：").strip()
    if not company_name:
        print("上市公司名称不能为空")
        return

    year_str = input("请输入年份（如2022）：").strip()
    year = validate_year(year_str)
    if year is None:
        print("年份无效，请输入2000-2030之间的年份")
        return

    print(f"\n正在查询 {company_name} {year}年年度报告...")

    result = get_mda(company_name, year)
    output = format_mda_output(result, company_name, year)

    print("\n" + output)


# if __name__ == "__main__":
#     main()


In [4]:
## 可视化


import pandas as pd
import os
import sys
import json

DATA_DIR = r"d:\软件\game\教学案例（北大）\data"
VALID_YEARS = [2022, 2023, 2024]
FINANCIAL_TYPES = {
    "资产负债表": "资产负债表",
    "利润表": "利润表",
    "现金流量表": "现金流量表"
}

BALANCE_KEYS = ["货币资金", "应收账款", "存货", "流动资产合计", "资产总计", "负债合计", "所有者权益合计"]
INCOME_KEYS = ["营业收入", "营业成本", "销售费用", "管理费用", "财务费用", "营业利润", "净利润"]
CASHFLOW_KEYS = ["销售商品、提供劳务收到的现金", "经营活动产生的现金流量净额", "投资活动产生的现金流量净额", "筹资活动产生的现金流量净额", "现金及现金等价物净增加额"]

_cache = {}

def get_file_path(year, financial_type):
    return os.path.join(DATA_DIR, f"{year}年{financial_type}.xlsx")

def load_financial_data(year, financial_type):
    cache_key = f"{year}_{financial_type}"
    if cache_key in _cache:
        return _cache[cache_key]
    file_path = get_file_path(year, financial_type)
    if not os.path.exists(file_path):
        return None
    df = pd.read_excel(file_path)
    _cache[cache_key] = df
    return df

def search_security(df, security_name):
    if df is None:
        return []
    matches = df[df['证券名称'] == security_name]
    return matches.index.tolist()

def get_data_for_company(df, row_idx, key_list):
    if df is None:
        return {}
    result = {}
    row = df.iloc[row_idx]
    col_mapping = {}
    for col in df.columns:
        col_name = str(col).split('\n')[0].strip()
        col_mapping[col_name] = col
    for key in key_list:
        if key in col_mapping:
            col = col_mapping[key]
            value = row[col]
            if pd.notna(value):
                try:
                    result[key] = round(float(value), 2)
                except:
                    result[key] = str(value)
    return result

def get_company_info(df, row_idx):
    if df is None:
        return None, None
    code = df.iloc[row_idx]['证券代码']
    name = df.iloc[row_idx]['证券名称']
    return str(code), str(name)

def load_company_data(security_name):
    results = {}
    for year in VALID_YEARS:
        results[year] = {}
        for ft, ft_name in FINANCIAL_TYPES.items():
            df = load_financial_data(year, ft_name)
            if df is None:
                continue
            indices = search_security(df, security_name)
            if len(indices) == 0:
                continue
            row_idx = indices[0]
            code, name = get_company_info(df, row_idx)
            if ft == "资产负债表":
                key_list = BALANCE_KEYS
            elif ft == "利润表":
                key_list = INCOME_KEYS
            else:
                key_list = CASHFLOW_KEYS
            data = get_data_for_company(df, row_idx, key_list)
            results[year][ft] = data
            if 'company_code' not in results:
                results['company_code'] = code
                results['company_name'] = name
    return results

def format_value(val):
    if val is None or val == '-':
        return '-'
    if isinstance(val, (int, float)):
        return f"{val:.2f}"
    return str(val)

def generate_chart_config(canvas_id, label, data_values, color):
    return {
        'type': 'line',
        'data': {
            'labels': ['2022', '2023', '2024'],
            'datasets': [{
                'label': label,
                'data': data_values,
                'borderColor': color,
                'backgroundColor': color + '20',
                'fill': True,
                'tension': 0.4
            }]
        },
        'options': {
            'responsive': True,
            'plugins': {'legend': {'display': False}},
            'scales': {'y': {'beginAtZero': False}}
        }
    }

def generate_html(data):
    company_name = data.get('company_name', '未知')
    company_code = data.get('company_code', '')

    balance_data = {}
    income_data = {}
    cashflow_data = {}

    for year in VALID_YEARS:
        if year in data:
            balance_data[year] = data[year].get('资产负债表', {})
            income_data[year] = data[year].get('利润表', {})
            cashflow_data[year] = data[year].get('现金流量表', {})

    def get_values(key, bdata, idata, cdata):
        vals = []
        for year in VALID_YEARS:
            val = bdata.get(year, {}).get(key) or idata.get(year, {}).get(key) or cdata.get(year, {}).get(key)
            if val is not None and val != '-':
                try:
                    vals.append(float(val))
                except:
                    vals.append(None)
            else:
                vals.append(None)
        return vals

    chart_configs = []
    
    def make_chart_box(key, canvas_id, color, bdata, idata, cdata):
        values = get_values(key, bdata, idata, cdata)
        config = generate_chart_config(canvas_id, key, values, color)
        chart_configs.append((canvas_id, config))
        return '<div class="chart-box"><div class="chart-title">' + key + '</div><canvas id="' + canvas_id + '"></canvas></div>'

    balance_charts = ''.join(make_chart_box(key, 'bs' + str(i), '#4a90d9', balance_data, {}, {}) for i, key in enumerate(BALANCE_KEYS))
    income_charts = ''.join(make_chart_box(key, 'is' + str(i), '#48bb78', {}, income_data, {}) for i, key in enumerate(INCOME_KEYS))
    cashflow_charts = ''.join(make_chart_box(key, 'cf' + str(i), '#ed8936', {}, {}, cashflow_data) for i, key in enumerate(CASHFLOW_KEYS))

    scripts = []
    for canvas_id, config in chart_configs:
        config_json = json.dumps(config, ensure_ascii=False)
        scripts.append(f'new Chart(document.getElementById("{canvas_id}"), {config_json});')
    all_scripts = '\n'.join(scripts)

    html = '''<!DOCTYPE html>
<html lang="zh-CN">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1.0">
<title>''' + company_name + ''' - 财务数据分析</title>
<style>
* {margin:0;padding:0;box-sizing:border-box}
body {font-family:'Microsoft YaHei',Arial,sans-serif;background:linear-gradient(135deg,#e6f3ff,#f0f8ff);min-height:100vh;padding:20px}
.container {max-width:1400px;margin:0 auto}
.header {background:linear-gradient(135deg,#4a90d9,#6ba3e0);color:#fff;padding:30px;border-radius:15px;margin-bottom:20px;box-shadow:0 4px 15px rgba(74,144,217,0.3)}
.header h1 {font-size:28px;margin-bottom:10px}
.header .subtitle {font-size:16px;opacity:0.9}
.section-title {font-size:24px;color:#2c5282;margin:30px 0 15px 0;padding-bottom:10px;border-bottom:2px solid #4a90d9}
.chart-grid {display:grid;grid-template-columns:repeat(auto-fill,minmax(400px,1fr));gap:20px}
.chart-box {background:#fff;border-radius:10px;padding:20px;box-shadow:0 2px 10px rgba(0,0,0,0.1)}
.chart-title {font-size:16px;font-weight:bold;color:#2c5282;margin-bottom:15px;text-align:center}
.chart-box canvas {max-height:250px}
</style>
</head>
<body>
<div class="container">
<div class="header">
<h1>''' + company_name + '''</h1>
<div class="subtitle">证券代码: ''' + company_code + ''' | 财务数据分析 (2022-2024)</div>
</div>
<h2 class="section-title">资产负债表</h2>
<div class="chart-grid">''' + balance_charts + '''</div>
<h2 class="section-title">利润表</h2>
<div class="chart-grid">''' + income_charts + '''</div>
<h2 class="section-title">现金流量表</h2>
<div class="chart-grid">''' + cashflow_charts + '''</div>
</div>
<script>
var cdnList = [
    'https://cdn.bootcdn.net/ajax/libs/Chart.js/4.4.1/chart.umd.min.js',
    'https://lib.baomitu.com/Chart.js/4.4.1/chart.umd.min.js',
    'https://cdn.jsdelivr.net/npm/chart.js@4.4.1/dist/chart.umd.min.js',
    'https://unpkg.com/chart.js@4.4.1/dist/chart.umd.min.js'
];
var cdnIndex = 0;
function loadChartJS() {
    if (typeof Chart !== 'undefined') {
        initCharts();
        return;
    }
    if (cdnIndex >= cdnList.length) {
        console.error('所有CDN都加载失败');
        alert('图表库加载失败，请检查网络连接后刷新页面');
        return;
    }
    var script = document.createElement('script');
    script.src = cdnList[cdnIndex];
    script.onload = function() {
        console.log('Chart.js 加载成功: ' + cdnList[cdnIndex]);
        initCharts();
    };
    script.onerror = function() {
        console.warn('CDN加载失败: ' + cdnList[cdnIndex]);
        cdnIndex++;
        loadChartJS();
    };
    document.head.appendChild(script);
}
function initCharts() {
    if (typeof Chart === 'undefined') {
        console.error('Chart.js 未加载');
        return;
    }
    try {
''' + all_scripts + '''
        console.log('图表初始化完成，共 ' + document.querySelectorAll('canvas').length + ' 个图表');
    } catch(e) {
        console.error('图表初始化失败:', e);
    }
}
if (document.readyState === 'loading') {
    document.addEventListener('DOMContentLoaded', loadChartJS);
} else {
    loadChartJS();
}
</script>
</body>
</html>'''
    return html

def main():
    if sys.stdout.encoding != 'utf-8':
        sys.stdout.reconfigure(encoding='utf-8')

    print("=" * 50)
    print("财务数据可视化页面生成器")
    print("=" * 50)

    if len(sys.argv) > 1:
        security_name = sys.argv[1].strip()
    else:
        security_name = input("请输入证券名称：").strip()

    if not security_name:
        print("证券名称不能为空")
        return

    print(f"\n正在查询 '{security_name}' 的财务数据...")
    data = load_company_data(security_name)

    if not data.get('company_name'):
        print(f"未找到证券名称为 '{security_name}' 的记录")
        return

    print(f"已找到: {data['company_name']} ({data['company_code']})")
    print("正在生成HTML页面...")

    html_content = generate_html(data)

    safe_name = ''.join(c for c in data['company_name'] if c.isalnum() or c in (' ', '-', '_'))
    safe_name = safe_name.strip()
    output_dir = r"d:\软件\game\教学案例（北大）\visualization"
    output_file = os.path.join(output_dir, f"财务可视化_{safe_name}.html")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(html_content)

    print(f"\n生成成功！")
    print(f"输出文件: {output_file}")
    print(f"\n请在浏览器中打开该文件查看可视化结果。")

    try:
        os.startfile(output_file)
    except:
        pass

# if __name__ == "__main__":
#     main()


In [5]:
"""上市公司财务分析智能体

功能：
1. 用户输入上市公司名称
2. 获取该上市公司近2022-2024年的财务三大表数据
3. 获取该上市公司近2022-2024年的管理层讨论与分析结果
4. 使用LLM对管理层讨论与分析进行解读，并基于管理层讨论与分析结果再对财务三大表数据进行整体解读
5. 生成MD和HTML格式报告

依赖:
    - get_financial_statement: 财务表查询
    - get_mda: 年报管理层讨论与分析查询
    - LLM: 大语言模型调用

使用方法:
    python agent.py
"""

import os
import sys
import json
import markdown
import concurrent.futures
from typing import Dict, List, Optional, Any
format_financial_output=format_output
# sys.path.append(os.path.dirname(__file__))

# from utils.get_financial_statement import (
#     load_financial_data,
#     search_security,
#     format_output as format_financial_output,
#     FINANCIAL_TYPES
# )
# from utils.get_mda import get_mda, format_mda_output
# from utils.LLM import chat

DATA_DIR = r"/usr/local/data"
OUTPUT_DIR = r"./output"
os.makedirs(OUTPUT_DIR,exist_ok=True)
VALID_YEARS = [2022, 2023, 2024]


class FinancialAnalysisAgent:
    """财务分析智能体"""

    def __init__(self, company_name: str):
        self.company_name = company_name
        self.financial_data = {}
        self.mda_data = {}
        self.llm_analysis = {}
        self.key_metrics = {}
        self.total_tokens = 0

    def get_financial_data(self) -> Dict[int, Dict[str, Any]]:
        """获取财务三大表数据"""
        print(f"\n[Step 1/4] 正在获取财务数据...")

        success_count = 0
        for year in VALID_YEARS:
            self.financial_data[year] = {}
            for fin_type in ["资产负债表", "利润表", "现金流量表"]:
                df = load_financial_data(year, fin_type)
                if df is not None:
                    indices = search_security(df, self.company_name)
                    if indices:
                        row_idx = indices[0]
                        self.financial_data[year][fin_type] = {
                            "data": df,
                            "row_idx": row_idx,
                            "formatted": self._extract_key_metrics(df, row_idx, fin_type)
                        }
                        print(f"  ✓ {year}年 {fin_type}")
                        success_count += 1
                    else:
                        print(f"  × {year}年 {fin_type} - 未找到")
                else:
                    print(f"  × {year}年 {fin_type} - 数据文件不存在")

        print(f"  财务数据获取完成！共获取 {success_count} 个财务报表")
        return self.financial_data

    def get_mda_data(self) -> Dict[int, str]:
        """获取管理层讨论与分析数据"""
        print(f"\n[Step 2/4] 正在获取管理层讨论与分析(MDA)...")

        auth = os.environ.get("MCP_AUTHORIZATION")
        if not auth:
            print("  警告: 未设置 MCP_AUTHORIZATION 环境变量，MDA数据可能无法获取")
            return {}

        success_count = 0
        for year in VALID_YEARS:
            try:
                result = get_mda(self.company_name, year, authorization=auth)
                formatted = format_mda_output(result, self.company_name, year)
                self.mda_data[year] = formatted
                print(f"  ✓ {year}年 MDA")
                success_count += 1
            except Exception as e:
                self.mda_data[year] = f"获取失败: {str(e)}"
                print(f"  × {year}年 MDA - 获取失败: {e}")

        print(f"  MDA数据获取完成！共获取 {success_count} 个")
        return self.mda_data

    def analyze_with_llm(self) -> Dict[str, str]:
        """使用LLM进行分析解读（并发执行）"""
        print("\n[Step 3/4] 正在进行LLM智能分析...")

        print("  正在准备数据...")
        mda_text = self._prepare_mda_text()
        balance_sheet_data = self._prepare_financial_text("资产负债表")
        income_data = self._prepare_financial_text("利润表")
        cash_flow_data = self._prepare_financial_text("现金流量表")
        print("  ✓ MDA数据准备完成")
        print("  ✓ 资产负债表数据准备完成")
        print("  ✓ 利润表数据准备完成")
        print("  ✓ 现金流量表数据准备完成")
        print("  正在调用LLM进行4项分析（并发执行）...")

        with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
            future_to_task = {
                executor.submit(self._llm_analyze_mda, mda_text): ("mda", "MDA解读"),
                executor.submit(self._llm_analyze_balance_sheet, balance_sheet_data, mda_text): ("balance_sheet", "资产负债表解读"),
                executor.submit(self._llm_analyze_income, income_data, mda_text): ("income", "利润表解读"),
                executor.submit(self._llm_analyze_cash_flow, cash_flow_data, mda_text): ("cash_flow", "现金流量表解读")
            }

            completed = 0
            total = len(future_to_task)
            for future in concurrent.futures.as_completed(future_to_task):
                task_key, task_display = future_to_task[future]
                completed += 1
                try:
                    result = future.result()
                    content = result["content"]
                    tokens = result.get("total_tokens", 0)
                    self.total_tokens += tokens
                    self.llm_analysis[task_key] = content
                    print(f"    [{completed}/{total}] {task_display} - 完成 (Token: {tokens:,})")
                except Exception as e:
                    self.llm_analysis[task_key] = f"分析失败: {str(e)}"
                    print(f"    [{completed}/{total}] {task_display} - 失败: {e}")

        print(f"  LLM分析完成！Token总消耗: {self.total_tokens:,}")
        self.key_metrics = self._extract_all_key_metrics()

        return self.llm_analysis

    def _prepare_mda_text(self) -> str:
        """准备MDA文本"""
        texts = []
        for year in VALID_YEARS:
            if year in self.mda_data:
                texts.append(f"=== {year}年 ===\n{self.mda_data[year]}")
        return "\n\n".join(texts) if texts else "暂无MDA数据"

    def _prepare_financial_text(self, fin_type: str) -> str:
        """准备财务数据文本"""
        texts = []
        for year in VALID_YEARS:
            if year in self.financial_data and fin_type in self.financial_data[year]:
                data = self.financial_data[year][fin_type]
                if data:
                    company_row = data["data"].iloc[data["row_idx"]]
                    formatted = format_financial_output(company_row)
                    texts.append(f"=== {year}年 {fin_type} ===\n{formatted}")
        return "\n\n".join(texts) if texts else f"暂无{fin_type}数据"

    def _extract_key_metrics(self, df, row_idx: int, fin_type: str) -> Dict[str, Any]:
        """提取关键财务指标"""
        if df is None or row_idx < 1:
            return {}

        metrics = {}
        header_row = df.iloc[0]

        for col_idx in range(2, min(len(df.columns), len(header_row))):
            header = str(header_row.iloc[col_idx]).split('\n')[0].strip()
            value = df.iloc[row_idx, col_idx]

            if fin_type == "资产负债表":
                if "资产" in header and "总计" in header:
                    metrics["总资产"] = value
                elif "负债" in header and "总计" in header:
                    metrics["总负债"] = value
                elif "归属" in header and "权益" in header:
                    metrics["归属于母公司股东权益"] = value
                elif "净利润" in header:
                    metrics["净利润"] = value

            elif fin_type == "利润表":
                if "营业收入" in header or "营业总收入" in header:
                    metrics["营业收入"] = value
                elif "净利润" in header:
                    metrics["净利润"] = value
                elif "毛利率" in header:
                    metrics["毛利率"] = value
                elif "净资产收益率" in header or "ROE" in header:
                    metrics["净资产收益率"] = value

            elif fin_type == "现金流量表":
                if "经营活动" in header and "产生的现金流量净额" in header:
                    metrics["经营活动产生的现金流量净额"] = value
                elif "投资活动" in header and "产生的现金流量净额" in header:
                    metrics["投资活动产生的现金流量净额"] = value
                elif "筹资活动" in header and "产生的现金流量净额" in header:
                    metrics["筹资活动产生的现金流量净额"] = value
                elif "现金及现金等价物净增加额" in header:
                    metrics["现金及现金等价物净增加额"] = value

        return metrics

    def _llm_analyze_mda(self, mda_text: str) -> Dict:
        """LLM分析MDA"""
        prompt = f"""请分析以下上市公司{self.company_name}的管理层讨论与分析(MDA)内容，要求：
1. 总结公司近3年的业务发展情况和经营亮点
2. 分析公司面临的主要风险和挑战
3. 突出管理层对未来发展的展望
4. 内容要精简、重点突出
5. 字数控制在500-1000字之间
6. 直接输出分析内容，不要有开头客套话（如"好的"、"作为一名专业的金融分析师"等）
7. 不要输出类似"XXX管理层讨论与分析（MDA）核心要点分析"等标题

MDA内容：
{mda_text}"""

        try:
            result = chat(
                system_prompt="你是一位专业的金融分析师，擅长分析上市公司年报和财务数据。",
                user_message=prompt,
                temperature=0.7
            )
            if result and result.get("content"):
                return result
            return {"content": "LLM分析结果为空", "total_tokens": 0}
        except Exception as e:
            return {"content": f"LLM分析失败: {str(e)}", "total_tokens": 0}

    def _llm_analyze_balance_sheet(self, balance_data: str, mda_text: str) -> Dict:
        """LLM分析资产负债表"""
        prompt = f"""请分析以下上市公司{self.company_name}的资产负债表数据，要求：
1. 分析总资产、总负债的变化趋势
2. 分析资产负债率的水平及变化
3. 结合管理层讨论与分析内容，解读财务状况
4. 突出重点指标的变化
5. 内容精简、观点明确
6. 字数控制在500-1000字之间
7. 直接输出分析内容，不要有开头客套话
8. 不要输出类似"XXX资产负债表分析报告"等标题
9. 财务数据保留2位小数

资产负债表数据：
{balance_data}

管理层讨论与分析：
{mda_text}"""

        try:
            result = chat(
                system_prompt="你是一位专业的金融分析师，擅长分析上市公司年报和财务数据。",
                user_message=prompt,
                temperature=0.7
            )
            if result and result.get("content"):
                return result
            return {"content": "LLM分析结果为空", "total_tokens": 0}
        except Exception as e:
            return {"content": f"LLM分析失败: {str(e)}", "total_tokens": 0}

    def _llm_analyze_income(self, income_data: str, mda_text: str) -> Dict:
        """LLM分析利润表"""
        prompt = f"""请分析以下上市公司{self.company_name}的利润表数据，要求：
1. 分析营业收入、净利润的变化趋势
2. 分析毛利率、净利率的水平及变化
3. 结合管理层讨论与分析内容，解读盈利情况
4. 突出重点指标的变化
5. 内容精简、观点明确
6. 字数控制在500-1000字之间
7. 直接输出分析内容，不要有开头客套话
8. 不要输出类似"XXX盈利情况分析报告"等标题
9. 财务数据保留2位小数

利润表数据：
{income_data}

管理层讨论与分析：
{mda_text}"""

        try:
            result = chat(
                system_prompt="你是一位专业的金融分析师，擅长分析上市公司年报和财务数据。",
                user_message=prompt,
                temperature=0.7
            )
            if result and result.get("content"):
                return result
            return {"content": "LLM分析结果为空", "total_tokens": 0}
        except Exception as e:
            return {"content": f"LLM分析失败: {str(e)}", "total_tokens": 0}

    def _llm_analyze_cash_flow(self, cash_flow_data: str, mda_text: str) -> Dict:
        """LLM分析现金流量表"""
        prompt = f"""请分析以下上市公司{self.company_name}的现金流量表数据，要求：
1. 分析经营活动、投资活动、筹资活动现金流的状况
2. 分析现金及现金等价物净增加额的变化
3. 结合管理层讨论与分析内容，解读现金流情况
4. 突出重点指标的变化
5. 内容精简、观点明确
6. 字数控制在500-1000字之间
7. 直接输出分析内容，不要有开头客套话
8. 不要输出类似"XXX现金流量分析报告"等标题
9. 财务数据保留2位小数

现金流量表数据：
{cash_flow_data}

管理层讨论与分析：
{mda_text}"""

        try:
            result = chat(
                system_prompt="你是一位专业的金融分析师，擅长分析上市公司年报和财务数据。",
                user_message=prompt,
                temperature=0.7
            )
            if result and result.get("content"):
                return result
            return {"content": "LLM分析结果为空", "total_tokens": 0}
        except Exception as e:
            return {"content": f"LLM分析失败: {str(e)}", "total_tokens": 0}

    def _extract_all_key_metrics(self) -> Dict[str, List[Dict[str, Any]]]:
        """提取所有关键指标用于可视化"""
        metrics = {
            "balance_sheet": {
                "总资产": [],
                "总负债": [],
                "净资产": []
            },
            "income": {
                "营业收入": [],
                "净利润": [],
                "毛利率": []
            },
            "cash_flow": {
                "经营活动产生的现金流量净额": [],
                "投资活动产生的现金流量净额": [],
                "筹资活动产生的现金流量净额": []
            }
        }

        for year in VALID_YEARS:
            if year in self.financial_data:
                bs_data = self.financial_data[year].get("资产负债表", {})
                income_data = self.financial_data[year].get("利润表", {})
                cf_data = self.financial_data[year].get("现金流量表", {})

                formatted_bs = bs_data.get("formatted", {})
                formatted_income = income_data.get("formatted", {})
                formatted_cf = cf_data.get("formatted", {})

                metrics["balance_sheet"]["总资产"].append({"year": year, "value": formatted_bs.get("总资产")})
                metrics["balance_sheet"]["总负债"].append({"year": year, "value": formatted_bs.get("总负债")})

                total_assets = formatted_bs.get("总资产")
                total_liabilities = formatted_bs.get("总负债")
                if total_assets and total_liabilities:
                    try:
                        net_assets = float(total_assets) - float(total_liabilities)
                        metrics["balance_sheet"]["净资产"].append({"year": year, "value": net_assets})
                    except (ValueError, TypeError):
                        metrics["balance_sheet"]["净资产"].append({"year": year, "value": None})

                metrics["income"]["营业收入"].append({"year": year, "value": formatted_income.get("营业收入")})
                metrics["income"]["净利润"].append({"year": year, "value": formatted_income.get("净利润")})
                metrics["income"]["毛利率"].append({"year": year, "value": formatted_income.get("毛利率")})

                metrics["cash_flow"]["经营活动产生的现金流量净额"].append({"year": year, "value": formatted_cf.get("经营活动产生的现金流量净额")})
                metrics["cash_flow"]["投资活动产生的现金流量净额"].append({"year": year, "value": formatted_cf.get("投资活动产生的现金流量净额")})
                metrics["cash_flow"]["筹资活动产生的现金流量净额"].append({"year": year, "value": formatted_cf.get("筹资活动产生的现金流量净额")})

        return metrics

    def generate_md_report(self, output_path: Optional[str] = None) -> Dict[str, str]:
        """生成MD格式报告和HTML格式报告"""
        print(f"\n[Step 4/4] 正在生成报告...")

        if output_path is None:
            md_output_path = os.path.join(OUTPUT_DIR, f"财务分析报告_{self.company_name}.md")
        else:
            md_output_path = output_path

        md_content = self._build_md_content()

        with open(md_output_path, 'w', encoding='utf-8') as f:
            f.write(md_content)

        print(f"  ✓ MD报告已生成")

        html_output_path = self._convert_md_to_html(md_output_path)
        print(f"  ✓ HTML报告已生成")

        return {"md": md_output_path, "html": html_output_path}

    def _convert_md_to_html(self, md_file_path: str) -> str:
        """将Markdown文件转换为HTML页面"""
        html_file_path = md_file_path.replace('.md', '_doc.html')

        with open(md_file_path, 'r', encoding='utf-8') as f:
            md_content = f.read()

        html_content = markdown.markdown(
            md_content,
            extensions=['tables', 'fenced_code', 'nl2br']
        )

        title = os.path.basename(md_file_path).replace('.md', '')

        html_template = f"""<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title}</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}
        body {{
            font-family: "Microsoft YaHei", "PingFang SC", "Hiragino Sans GB", "Arial", sans-serif;
            font-size: 16px;
            line-height: 1.8;
            color: #000;
            background-color: #fff;
            padding: 20px;
        }}
        .container {{
            max-width: 900px;
            margin: 0 auto;
            background-color: #fff;
            padding: 40px;
            border: 1px solid #ccc;
            border-radius: 0;
        }}
        h1 {{
            font-size: 28px;
            color: #000;
            text-align: center;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 2px solid #000;
        }}
        h2 {{
            font-size: 22px;
            color: #000;
            margin-top: 35px;
            margin-bottom: 20px;
            padding-left: 15px;
            border-left: 3px solid #000;
        }}
        h3 {{
            font-size: 18px;
            color: #000;
            margin-top: 25px;
            margin-bottom: 15px;
        }}
        p {{
            margin-bottom: 15px;
            text-align: justify;
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
            font-size: 14px;
        }}
        th, td {{
            border: 1px solid #000;
            padding: 12px;
            text-align: left;
            vertical-align: top;
        }}
        th {{
            background-color: #000;
            color: #fff;
            font-weight: bold;
            text-align: center;
        }}
        tr:nth-child(even) {{
            background-color: #f0f0f0;
        }}
        tr:hover {{
            background-color: #e0e0e0;
        }}
        ul, ol {{
            margin-bottom: 15px;
            padding-left: 30px;
        }}
        li {{
            margin-bottom: 8px;
        }}
        hr {{
            border: none;
            border-top: 1px solid #000;
            margin: 30px 0;
        }}
    </style>
</head>
<body>
    <div class="container">
{html_content}
    </div>
</body>
</html>"""

        with open(html_file_path, 'w', encoding='utf-8') as f:
            f.write(html_template)

        print(f"HTML报告已生成: {html_file_path}")
        return html_file_path

    def _build_md_content(self) -> str:
        """构建MD报告内容"""
        lines = []

        lines.append(f"# {self.company_name} 财务分析报告")
        lines.append("")
        lines.append(f"**分析期间：2022-2024年**")
        lines.append("")
        lines.append("---")
        lines.append("")

        lines.append("## 一、管理层讨论与分析解读")
        lines.append("")
        lines.append(self.llm_analysis.get("mda", "暂无"))
        lines.append("")

        lines.append("## 二、资产负债表解读")
        lines.append("")
        lines.append(self.llm_analysis.get("balance_sheet", "暂无"))
        lines.append("")

        lines.append("## 三、利润表解读")
        lines.append("")
        lines.append(self.llm_analysis.get("income", "暂无"))
        lines.append("")

        lines.append("## 四、现金流量表解读")
        lines.append("")
        lines.append(self.llm_analysis.get("cash_flow", "暂无"))
        lines.append("")

        lines.append("---")
        lines.append("")
        lines.append("*本报告由财务分析智能体自动生成*")

        return "\n".join(lines)


def run_analysis(company_name: str) -> Dict[str, str]:
    """运行财务分析"""
    import time
    start_time = time.time()

    print("=" * 60)
    print(f"开始分析: {company_name}")
    print("=" * 60)

    agent = FinancialAnalysisAgent(company_name)

    agent.get_financial_data()

    agent.get_mda_data()

    agent.analyze_with_llm()

    md_result = agent.generate_md_report()

    elapsed_time = time.time() - start_time
    print(f"\n{'=' * 60}")
    print(f"分析完成！总耗时: {elapsed_time:.1f}秒 | Token总消耗: {agent.total_tokens:,}")
    print(f"MD文档: {md_result['md']}")
    print(f"HTML报告: {md_result['html']}")
    print("=" * 60)

    return {"md": md_result["md"], "md_html": md_result["html"]}


def main():
    """主函数"""
    print("=" * 60)
    print("上市公司财务分析智能体")
    print("=" * 60)

    company_name = input("请输入上市公司名称（如：同花顺）：").strip()

    if not company_name:
        print("公司名称不能为空")
        return

    try:
        run_analysis(company_name)
    except Exception as e:
        print(f"\n分析过程中出现错误: {str(e)}")
        import traceback
        traceback.print_exc()


if __name__ == "__main__":
    main()


上市公司财务分析智能体


请输入上市公司名称（如：同花顺）： 紫金矿业


开始分析: 紫金矿业

[Step 1/4] 正在获取财务数据...
  ✓ 2022年 资产负债表
  ✓ 2022年 利润表
  ✓ 2022年 现金流量表
  ✓ 2023年 资产负债表
  ✓ 2023年 利润表
  ✓ 2023年 现金流量表
  ✓ 2024年 资产负债表
  ✓ 2024年 利润表
  ✓ 2024年 现金流量表
  财务数据获取完成！共获取 9 个财务报表

[Step 2/4] 正在获取管理层讨论与分析(MDA)...
  ✓ 2022年 MDA
  ✓ 2023年 MDA
  ✓ 2024年 MDA
  MDA数据获取完成！共获取 3 个

[Step 3/4] 正在进行LLM智能分析...
  正在准备数据...
  ✓ MDA数据准备完成
  ✓ 资产负债表数据准备完成
  ✓ 利润表数据准备完成
  ✓ 现金流量表数据准备完成
  正在调用LLM进行4项分析（并发执行）...
    [1/4] 利润表解读 - 完成 (Token: 7,793)
    [2/4] 资产负债表解读 - 完成 (Token: 9,243)
    [3/4] MDA解读 - 完成 (Token: 7,216)
    [4/4] 现金流量表解读 - 完成 (Token: 9,784)
  LLM分析完成！Token总消耗: 34,036

[Step 4/4] 正在生成报告...
  ✓ MD报告已生成
HTML报告已生成: ./output/财务分析报告_紫金矿业_doc.html
  ✓ HTML报告已生成

分析完成！总耗时: 71.4秒 | Token总消耗: 34,036
MD文档: ./output/财务分析报告_紫金矿业.md
HTML报告: ./output/财务分析报告_紫金矿业_doc.html
